### Finding A1: Value at Risk and CVaR
On a typical bad day (the worst 5% of trading days), equity funds lost about 1.7% on average (VaR), and about 2.1% on the very worst of those days (CVaR). Debt funds lost about 0.2% and 0.3% respectively. Small-cap funds carry the largest tail risk: the five funds with the highest VaR are all small-cap schemes, with VaR of about -2.5% to -2.7% a day. CVaR is always worse than VaR, which shows that losses beyond the 5% threshold are substantially larger than the threshold itself.

### Finding A2: Rolling 90-day Sharpe ratio
Rolling Sharpe ratios swung widely for all four funds, mostly between about -2 and +4. Mirae Asset Large Cap had the highest average (1.43) over the period, peaking near 6 in early 2025, but fell to about -0.07 by May 2026. ICICI Pru Midcap ended strongest at 2.21, while HDFC Mid-Cap (1.69) recovered strongly from weak periods in 2024. The liquid fund averaged only 0.32 and also swung between about -3 and +3, which is implausible for a low-risk fund and suggests the dataset's NAV series are partly synthetic. No fund held the top position throughout, so a single period's ranking is a weak guide to future risk-adjusted returns.

### Finding A3: Investor cohorts
Almost all investors (4,803 of 5,000, about 96%) first transacted in 2024, because the transaction records start in January 2024. The 197 investors who started in 2025 had a higher average SIP (about Rs. 13,500 against Rs. 11,000 for the 2024 cohort), but the group is small, so this difference may be chance. Debt funds took about 16% of the money in both cohorts. Because the data has no pre-2024 cohorts, the analysis cannot show how investors' behaviour changes over time, and this limitation should be stated in the report.

# One check worth running

Before accepting the result, check how regular the gaps are. Paste this into a new cell in the notebook, or save it as a script:

# python program


sip_gaps = sip.dropna(subset=["gap_days"])
print(sip_gaps["gap_days"].describe().round(1))
print(sip_gaps["gap_days"].round(-1).value_counts().sort_index().head(10))


# If most gaps cluster near 30 days, the average is being pulled up by a few very long gaps, and the 35-day rule is still meaningful. If they’re spread widely, the SIP dates are random, and we report the at-risk share as a limitation rather than a result.

### Finding A4: SIP continuation
Of the 1,362 investors with at least six SIPs, 1,332 (97.8%) have an average gap between payments above 35 days. The gaps between consecutive SIPs are widely spread, from 0 to 494 days, with a median of 60 days. Real monthly SIPs would cluster near 30 days, and the dataset also contains 913 same-day SIP pairs, which do not occur in practice. The at-risk flag therefore reflects the simulated transaction dates rather than investor behaviour. It is reported here as a limitation of the dataset, and the threshold should be reviewed if real transaction records are used.

In [3]:
import sqlite3
from pathlib import Path
import pandas as pd

BASE = Path("..").resolve()
conn = sqlite3.connect(BASE / "Data" / "db" / "bluestock_mf.db")
sip = pd.read_sql_query(
    "SELECT investor_id, transaction_date FROM fact_transactions "
    "WHERE transaction_type = 'SIP'",
    conn, parse_dates=["transaction_date"]
)
conn.close()

sip = sip.sort_values(["investor_id", "transaction_date"])
sip["gap_days"] = sip.groupby("investor_id")["transaction_date"].diff().dt.days
sip_gaps = sip.dropna(subset=["gap_days"])

print(sip_gaps["gap_days"].describe().round(1))
print()
print(sip_gaps["gap_days"].round(-1).value_counts().sort_index().head(10))

count    14954.0
mean        83.1
std         77.9
min          0.0
25%         25.0
50%         60.0
75%        117.0
max        494.0
Name: gap_days, dtype: float64

gap_days
0.0      913
10.0    1385
20.0    1517
30.0    1114
40.0    1177
50.0     903
60.0     981
70.0     738
80.0     760
90.0     540
Name: count, dtype: int64


### Finding A5: Sector concentration (HHI)
The Herfindahl-Hirschman Index (HHI) was calculated for 34 equity funds from their sector weights. The average HHI is about 2,029 and the median 2,042, so a typical fund puts most of its money into a few sectors rather than spreading it evenly. Axis Bluechip (2,968) is the most concentrated, with 7 sectors, while UTI Mid Cap (1,240) is the least concentrated, spread across 10 sectors. Concentration is driven mainly by the number of sectors held, since funds with more sectors score lower. Each fund holds only about 6 to 10 stocks in this dataset, which is far fewer than real equity funds, so these figures should be read as a comparison between funds rather than as exact measures of real portfolios.

### Finding A6: Fund recommender
The recommender selects the top three funds by calculated Sharpe ratio within each risk grade. For Low risk, the liquid funds ICICI Pru Liquid (Sharpe 0.96) and Kotak Liquid (0.32) rank first and second, and SBI Magnum Gilt is third with a negative Sharpe (-0.21), meaning it returned less than the 6.5% risk-free rate. For Moderate risk, Mirae Asset Large Cap (1.64) leads, and for High risk, Mirae Asset Tax Saver (1.37). Very High risk is led by DSP Small Cap (0.99). The output is rule-based, not personalised advice. It depends on NAV-based Sharpe ratios calculated from a dataset with limited realism, so it should be read as a demonstration of the method rather than a recommendation to invest.